# Métricas de ML  
Alumno  
Marcelo Ismael López Verdugo -------||     A00959089  

## Sesión spark y setup  

### Librerías y sesión  

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, when, isnan, lit, desc, hour, dayofmonth, dayofweek
from pyspark.sql.functions import split, regexp_extract
from pyspark.sql.functions import collect_set, countDistinct
from pyspark.sql.types import DoubleType, FloatType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor, GBTRegressor
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator




spark = SparkSession.builder \
    .appName("Analisis Ecommerce Octubre") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

spark

In [2]:
import os
os.environ['JAVA_HOME'] 

'C:\\Program Files\\Eclipse Adoptium\\jdk-17.0.18.8-hotspot'

### Directorio del dataset  

In [3]:
root=".."
file_path = root + "/datasets/2019-Oct.csv"

df = spark.read.option("header", True) \
               .option("inferSchema", True) \
               .csv(file_path)

df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-09-30 17:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-09-30 17:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-09-30 17:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-09-30 17:0

In [4]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



### Depuración y preparación para ML

In [26]:
df_clean = df.dropna()
df_clean = df_clean.withColumn("weekhour", dayofweek(col("event_time")) + hour(col("event_time")) / 24)

non_numeric_columns = [name for name, dtype in df_clean.dtypes if dtype == "string"]
for col_name in non_numeric_columns:
    distinct_count = df_clean.select(col_name).distinct().count()
    print(f"{col_name}: dtype=string, distinct={distinct_count}")

    top_values = [row[col_name] for row in df_clean.groupBy(col_name)
                                              .count()
                                              .orderBy(desc("count"))
                                              .limit(10)
                                              .select(col_name)
                                              .collect()]

    grouped_col = f"{col_name}_grouped"
    df_clean = df_clean.withColumn(
        grouped_col,
        when(col(col_name).isin(top_values), col(col_name)).otherwise(lit("other"))
    )

print("Columnas agrupadas creadas:", [f"{c}_grouped" for c in non_numeric_columns])

df_clean.describe().show()

event_type: dtype=string, distinct=3
category_code: dtype=string, distinct=126
brand: dtype=string, distinct=1731
user_session: dtype=string, distinct=6419693
Columnas agrupadas creadas: ['event_type_grouped', 'category_code_grouped', 'brand_grouped', 'user_session_grouped']
+-------+----------+-----------------+--------------------+-------------------+--------+-----------------+--------------------+--------------------+------------------+------------------+---------------------+-------------+--------------------+
|summary|event_type|       product_id|         category_id|      category_code|   brand|            price|             user_id|        user_session|          weekhour|event_type_grouped|category_code_grouped|brand_grouped|user_session_grouped|
+-------+----------+-----------------+--------------------+-------------------+--------+-----------------+--------------------+--------------------+------------------+------------------+---------------------+-------------+--------------

## 1. Construcción de muestra M representativa

In [27]:
#Dataset balanceado (muestreo por porcentaje de probabilidad de cada clase al 0.1%)
df_sample=df_clean.sampleBy("event_type", fractions={"view": 0.001, "cart": 0.001, "purchase": 0.001}, seed=42)
df_sample.show(15, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------+-------+-------+---------+------------------------------------+------------------+------------------+-----------------------------+-------------+--------------------+
|event_time         |event_type|product_id|category_id        |category_code                |brand  |price  |user_id  |user_session                        |weekhour          |event_type_grouped|category_code_grouped        |brand_grouped|user_session_grouped|
+-------------------+----------+----------+-------------------+-----------------------------+-------+-------+---------+------------------------------------+------------------+------------------+-----------------------------+-------------+--------------------+
|2019-09-30 19:40:27|view      |1004133   |2053013555631882655|electronics.smartphone       |xiaomi |135.99 |550308094|6ca60043-5b92-4827-ba8e-9759d47f6e68|2.7916666666666665|view              |electronics.smartphone    

Puesto que en un futuro se usará "brand" como input, se observa la proporción de representación de cada marca entre la muestra M y la población P

In [33]:
df_clean.groupBy("brand_grouped").count().show()
df.groupby("brand").count().show()

+-------------+--------+
|brand_grouped|   count|
+-------------+--------+
|        apple| 4092652|
|        other|11136278|
|         oppo|  482887|
|           lg|  508999|
|         acer|  428081|
|      samsung| 5158902|
|       lenovo|  337970|
|       huawei| 1092346|
|           hp|  295026|
|        bosch|  329835|
|       xiaomi| 2697644|
+-------------+--------+

+--------+------+
|   brand| count|
+--------+------+
|yokohama|115014|
|   welss|  2895|
| tuffoni|  2121|
|    tmnt|   209|
| serebro|  1402|
| edifier|  2452|
|   globo|    31|
|    tega|  1475|
|  vortex|    46|
|   sonel|  7189|
|nutricia|  1710|
| bombbar|   384|
|  alutec|  2655|
|   goo.n|  1005|
| keenway|  2075|
|   sigma|  1697|
|   fagor|    60|
|eurogold|   479|
|   lotos|    16|
|  marley|   894|
+--------+------+
only showing top 20 rows


In [34]:
df_grouped = df_sample.groupBy("weekhour","brand_grouped").agg(
    sum(when(col("event_type") == "view", col("price")).otherwise(0)).alias("view_total_price"),
    sum(when(col("event_type") == "cart", col("price")).otherwise(0)).alias("cart_total_price"),
    sum(when(col("event_type") == "purchase", col("price")).otherwise(0)).alias("purchase_total_price")
)

df_grouped.show(20, truncate=False)

+------------------+-------------+------------------+------------------+--------------------+
|weekhour          |brand_grouped|view_total_price  |cart_total_price  |purchase_total_price|
+------------------+-------------+------------------+------------------+--------------------+
|3.1666666666666665|lenovo       |767.56            |0.0               |0.0                 |
|3.25              |samsung      |16081.27          |1678.8            |173.46              |
|3.25              |xiaomi       |6168.469999999999 |0.0               |663.8               |
|3.2916666666666665|acer         |2301.4399999999996|0.0               |0.0                 |
|2.8333333333333335|other        |12855.240000000002|218.77            |76.32               |
|2.9166666666666665|xiaomi       |4414.43           |586.83            |183.48              |
|3.0416666666666665|samsung      |21156.190000000002|652.64            |0.0                 |
|3.3333333333333335|other        |24939.63          |238.1  

In [35]:
df_grouped.describe().show()

+-------+-----------------+-------------+------------------+------------------+--------------------+
|summary|         weekhour|brand_grouped|  view_total_price|  cart_total_price|purchase_total_price|
+-------+-----------------+-------------+------------------+------------------+--------------------+
|  count|             1597|         1597|              1597|              1597|                1597|
|   mean|4.471613441870176|         NULL|5587.2211458985585|195.07634940513464|  150.01621164683786|
| stddev|2.012936238556284|         NULL|  7734.10957730893|   540.71055050225|   449.7583314011809|
|    min|              1.0|         acer|               0.0|               0.0|                 0.0|
|    max|7.958333333333333|       xiaomi|          44651.72| 5496.539999999999|  4798.5599999999995|
+-------+-----------------+-------------+------------------+------------------+--------------------+



## 2.1 Modelo de regresión por árbol de decisión
Usaremos `weekhour`, `view_total_price`, `cart_total_price` como entradas numéricas y `brand` como variable categórica para predecir `purchase_total_price`.

In [36]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Indexar la variable categórica brand
brand_indexer = StringIndexer(inputCol="brand_grouped", outputCol="brand_indexed", handleInvalid="skip")

# Armar vector de features
assembler = VectorAssembler(
    inputCols=["weekhour", "view_total_price", "cart_total_price", "brand_indexed"],
    outputCol="features",
    handleInvalid="skip"
)

# Definir regresor de árbol de decisión
dt_regressor = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="purchase_total_price",
    maxDepth=5,
    seed=42
)

# Pipeline de entrenamiento
pipeline_ml = Pipeline(stages=[brand_indexer, assembler, dt_regressor])

# Dividir datos en entrenamiento y prueba
train_df, test_df = df_grouped.randomSplit([0.8, 0.2], seed=42)
print("Train count:", train_df.count())
print("Test count:", test_df.count())

# Entrenar el modelo
model_pipeline = pipeline_ml.fit(train_df)

# Predecir en el conjunto de prueba
predictions = model_pipeline.transform(test_df)

# Evaluar el modelo
rmse_evaluator = RegressionEvaluator(
    labelCol="purchase_total_price",
    predictionCol="prediction",
    metricName="rmse"
)
r2_evaluator = RegressionEvaluator(
    labelCol="purchase_total_price",
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(predictions)
r2 = r2_evaluator.evaluate(predictions)

print("RMSE:", rmse)
print("R2:", r2)

# Mostrar algunas predicciones
predictions.select("weekhour", "brand_grouped", "view_total_price", "cart_total_price", "purchase_total_price", "prediction").show(10, truncate=False)

Train count: 1323
Test count: 274
RMSE: 428.2973838461059
R2: 0.09904668825212914
+------------------+-------------+------------------+----------------+--------------------+------------------+
|weekhour          |brand_grouped|view_total_price  |cart_total_price|purchase_total_price|prediction        |
+------------------+-------------+------------------+----------------+--------------------+------------------+
|1.0               |bosch        |2028.52           |0.0             |0.0                 |11.193741007194243|
|1.0               |lg           |2843.73           |0.0             |0.0                 |11.193741007194243|
|1.0               |other        |17715.95          |0.0             |555.97              |203.5006349206349 |
|1.0416666666666667|bosch        |244.46            |0.0             |0.0                 |11.193741007194243|
|1.0416666666666667|other        |18737.420000000002|151.56          |231.64              |203.5006349206349 |
|1.0833333333333333|apple     

## 2.2 Modelo GBT Regressor

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor  # Excelente regresor para datos tabulares
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# Definir el regresor GBT
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="purchase_total_price",
    seed=42
)

# Pipeline base
pipeline_ml = Pipeline(stages=[brand_indexer, assembler, gbt])

# ==========================================
# 2. CONFIGURACIÓN DEL TUNING (Búsqueda de Hiperparámetros)
# ==========================================
# Creamos la grilla de parámetros a evaluar. 
# Accedemos a los parámetros del modelo 'gbt' usando su referencia.
param_grid = (ParamGridBuilder()
              .addGrid(gbt.maxDepth, [3, 5, 7])       # Profundidad del árbol
              .addGrid(gbt.maxIter, [10, 20])         # Número de iteraciones (árboles a construir)
              .addGrid(gbt.stepSize, [0.05, 0.1])     # Tasa de aprendizaje (learning rate)
              .build())

# Definir el evaluador que usará el CrossValidator para medir la calidad (RMSE)
evaluator = RegressionEvaluator(
    labelCol="purchase_total_price",
    predictionCol="prediction",
    metricName="rmse"
)

# Configurar el Validador Cruzado (CrossValidator)
# numFolds=3 dividirá tus datos de entrenamiento en 3 pliegues para validar cada combinación
cross_val = CrossValidator(
    estimator=pipeline_ml,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

# ==========================================
# 3. ENTRENAMIENTO Y EVALUACIÓN
# ==========================================
# Aprovechando datos en entrenamiento y prueba

print("Iniciando búsqueda de hiperparámetros con Validación Cruzada...")
cv_model = cross_val.fit(train_df)
print("¡Búsqueda finalizada!")

# Obtener el mejor modelo encontrado por el CrossValidator
best_pipeline = cv_model.bestModel
best_gbt_model = best_pipeline.stages[-1] # El GBTRegressor es el último stage del pipeline

print("\n Mejores Hiperparámetros encontrados:")
print(f" - maxDepth: {best_gbt_model.getOrDefault('maxDepth')}")
print(f" - maxIter: {best_gbt_model.getOrDefault('maxIter')}")
print(f" - stepSize: {best_gbt_model.getOrDefault('stepSize')}")

# Predecir en el conjunto de prueba usando el mejor modelo
predictions = cv_model.transform(test_df)

# Evaluar el mejor modelo en datos que nunca ha visto
rmse = evaluator.evaluate(predictions)
r2_evaluator = RegressionEvaluator(labelCol="purchase_total_price", predictionCol="prediction", metricName="r2")
r2 = r2_evaluator.evaluate(predictions)

print(f"\n Resultados en el Test Set:")
print("RMSE:", rmse)
print("R2:", r2)

# Mostrar algunas predicciones
predictions.select("weekhour", "brand", "view_total_price", "cart_total_price", "purchase_total_price", "prediction").show(10, truncate=False)


Iniciando búsqueda de hiperparámetros con Validación Cruzada...


Considerando la búsqueda de hiperparámetros se encontró que en GBTRegressor se pudo desempeñar con una búsqueda en grid.  Para ambos métodos de aprendizaje fue necesario generar una gestión de cardinalidad para el caso de la marca ("brand") donde originalmente hay presencia de muchas opciones y las de baja participación se juntaron en una sola como "other".  El manejo de tal cardinalidad es útil para distribuciones en estilo Pareto ya que la mayoría de la participación la darán los Top X.  Para otro tipo de muestras es posible desarrollar un método por estratificación donde se forman "bins" de datos para obtener un comportamiento típico esperado entre la muestra M y la población.